# Audio Retrieval — Stage 1: Upper Bound Baseline

**Single text encoder for both queries and corpus.**

| Stage | Query Encoding | Doc Encoding | Purpose |
|-------|---------------|--------------|------------------|
| 1 ✅ | Text → Encoder | Text → Encoder | **Upper bound** |
| 2 | ASR(Audio) → Encoder | Text → Encoder | Lower bound |
| 3 | Audio → AudioEncoder | Text → Encoder | Final system |

**Dataset:** `MedRAG/pubmed` (HuggingFace)  
**Metrics:** MRR, MAP, NDCG, Recall@1/5/10

In [ ]:
# ── Cell 1: Install dependencies ─────────────────────────────────────────
!pip install datasets transformers torch sentence-transformers

In [1]:
# ── Cell 2: Config — change only this cell ────────────────────────────────
CONFIG = {
    # Text encoder: swap this to try different models
    # Recommended options:
    #   "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract"
    #   "dmis-lab/biobert-base-cased-v1.2"
    #   "sentence-transformers/all-MiniLM-L6-v2"
    "encoder_name": "sentence-transformers/all-MiniLM-L6-v2",

    # How many corpus docs to use (None = full dataset, use small number to test)
    "corpus_size": 500,

    # Number of queries to evaluate (sampled from corpus titles)
    "num_queries": 200,

    # Retrieval depth
    "top_k": 10,

    # Encoding batch size — reduce if OOM on Kaggle T4
    "batch_size": 64,

    # Max token length
    "max_length": 512,

    # Output directory
    "output_dir": "/kaggle/working/results",
}

In [2]:
# ── Cell 3: Imports ───────────────────────────────────────────────────────
import json
import os
import random
from collections import defaultdict
from pathlib import Path

import numpy as np
import torch
from datasets import load_dataset
from tqdm.notebook import tqdm
from transformers import AutoModel, AutoTokenizer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}")
print(f"Encoder: {CONFIG['encoder_name']}")

os.makedirs(CONFIG["output_dir"], exist_ok=True)

Device : cuda
Encoder: sentence-transformers/all-MiniLM-L6-v2


In [3]:
print("Loading MedRAG/pubmed (streaming mode)...")
n = CONFIG["corpus_size"] or 1000

dataset = load_dataset("MedRAG/pubmed", split="train", streaming=True)
dataset = dataset.take(n)  # pulls only n rows, no full download

# streaming datasets are iterables, not indexable — convert to list
dataset = list(dataset)

print(f"Corpus size : {len(dataset):,} documents")
print(f"Columns     : {list(dataset[0].keys())}\n")
print("Sample document:")
for k, v in dataset[0].items():
    print(f"  {k}: {str(v)[:120]}")

Loading MedRAG/pubmed (streaming mode)...


Resolving data files:   0%|          | 0/1166 [00:00<?, ?it/s]

Corpus size : 500 documents
Columns     : ['id', 'title', 'content', 'contents', 'PMID']

Sample document:
  id: pubmed23n0001_0
  title: [Biochemical studies on camomile components/III. In vitro studies about the antipeptic activity of (--)-alpha-bisabolol 
  content: (--)-alpha-Bisabolol has a primary antipeptic action depending on dosage, which is not caused by an alteration of the pH
  contents: [Biochemical studies on camomile components/III. In vitro studies about the antipeptic activity of (--)-alpha-bisabolol 
  PMID: 21


In [4]:
# ── Cell 5: Build corpus, queries, qrels ─────────────────────────────────
#
# corpus  : { doc_id -> {title, content, contents} }
# queries : { qid    -> query_text }   <- title used as pseudo-query
# qrels   : { qid    -> {doc_id: 1} }  <- gold = the doc the title came from
#
# NOTE: Using title as pseudo-query is a clean upper-bound proxy.
#       In Stage 2 you will replace query_text with ASR(audio_of_title).

corpus = {}
for row in dataset:
    doc_id = str(row["id"])
    title = (row.get("title") or "").strip()
    content = (row.get("content") or "").strip()
    contents = (row.get("contents") or f"{title} {content}").strip()
    corpus[doc_id] = {
        "title"   : title,
        "content" : content,
        "contents": contents,
    }

# Sample queries from corpus
all_doc_ids = list(corpus.keys())
random.seed(42)
query_doc_ids = random.sample(all_doc_ids, min(CONFIG["num_queries"], len(all_doc_ids)))

queries = {}
qrels   = {}
for doc_id in query_doc_ids:
    qid = f"q_{doc_id}"
    queries[qid] = corpus[doc_id]["title"]   # <── swap with ASR text in Stage 2
    qrels[qid]   = {doc_id: 1}

print(f"Corpus docs : {len(corpus):,}")
print(f"Queries     : {len(queries):,}")
print(f"Qrels       : {sum(len(v) for v in qrels.values()):,} judgments")

Corpus docs : 500
Queries     : 200
Qrels       : 200 judgments


In [5]:
# ── CELL TTS: Text → Audio (Piper TTS) ───────────────────────────────────
!pip install -q piper-tts
!wget -q https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/lessac/low/en_US-lessac-low.onnx
!wget -q https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/lessac/low/en_US-lessac-low.onnx.json

import os
import wave
from piper import PiperVoice
from tqdm.notebook import tqdm

AUDIO_DIR = "/kaggle/working/audio"
os.makedirs(AUDIO_DIR, exist_ok=True)

voice = PiperVoice.load("en_US-lessac-low.onnx", use_cuda=(DEVICE == "cuda"))

def synthesize_and_save(query, qid, target_sr=16000):
    path = os.path.join(AUDIO_DIR, f"{qid}.wav")
    if os.path.exists(path):
        return path
    with wave.open(path, "wb") as wav_file:
        voice.synthesize_wav(query, wav_file)
    return path

# Generate audio for all queries
print(f"Generating audio for {len(queries):,} queries...")
query_audio = {}  # { qid -> wav_path }

for qid, text in tqdm(queries.items(), desc="TTS"):
    query_audio[qid] = synthesize_and_save(text, qid)

print(f"Done. Sample: {list(query_audio.items())[0]}")

/usr/local/lib/python3.12/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


Generating audio for 200 queries...


TTS:   0%|          | 0/200 [00:00<?, ?it/s]

Done. Sample: ('q_pubmed23n0001_327', '/kaggle/working/audio/q_pubmed23n0001_327.wav')


In [6]:
# ── CELL ASR: Audio → Text (Whisper-tiny) ────────────────────────────────
# Transcribes each .wav back to text — intentionally weak (whisper-tiny)
# Output: asr_queries = { qid -> transcribed_text }
# Drop-in replacement for `queries` in Cell 9 to get the lower bound
 
!pip install -q openai-whisper
 
import whisper
 
print("Loading Whisper-tiny ...")
asr_model = whisper.load_model("tiny", device=DEVICE)
print("  Done.")
 
def audio_to_text(wav_path: str) -> str:
    """
    Transcribe a .wav file using Whisper-tiny.
    fp16=False for Kaggle CPU fallback safety.
    """
    result = asr_model.transcribe(
        wav_path,
        language="en",
        fp16=(DEVICE == "cuda"),
    )
    return result["text"].strip()
 
 
# Transcribe all query audio
print(f"Transcribing {len(query_audio):,} audio files...")
asr_queries = {}  # { qid -> transcribed_text }
 
for qid, wav_path in tqdm(query_audio.items(), desc="ASR"):
    asr_queries[qid] = audio_to_text(wav_path)
 
print("\nSample transcriptions (original → ASR):")
for qid in list(asr_queries.keys())[:3]:
    print(f"  ORIGINAL : {queries[qid][:80]}")
    print(f"  ASR      : {asr_queries[qid][:80]}")
    print()


Loading Whisper-tiny ...
  Done.
Transcribing 200 audio files...


ASR:   0%|          | 0/200 [00:00<?, ?it/s]


Sample transcriptions (original → ASR):
  ORIGINAL : Papillary necrosis associated with calyceal arteritis.
  ASR      : Peppellerina Crosis associated with Colisell Art Arritis.

  ORIGINAL : The behavior of holo- and apo-forms of bovine superoxide dismutase at low pH.
  ASR      : The behavior of holo and the poforms of bovine superoxide dismutates at low pH.

  ORIGINAL : Poly(8-aminoguanylic acid): formation of ordered self-structures and interaction
  ASR      : Poly-8emenigu and Iliq acid, formation of ordered self-structures and interactio



In [7]:
# ── Cell 6: Load encoder ──────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(CONFIG["encoder_name"])
model     = AutoModel.from_pretrained(CONFIG["encoder_name"]).to(DEVICE)
model.eval()
print(f"Model loaded: {CONFIG['encoder_name']}")
print(f"Parameters  : {sum(p.numel() for p in model.parameters()):,}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded: sentence-transformers/all-MiniLM-L6-v2
Parameters  : 22,713,216


In [8]:
# ── Cell 7: Encoding utilities ────────────────────────────────────────────
def mean_pool(token_embeddings, attention_mask):
    """Mean pool over non-padding tokens."""
    mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * mask, dim=1) / torch.clamp(mask.sum(dim=1), min=1e-9)


def encode(texts, desc="Encoding"):
    """
    Encode a list of strings → L2-normalised embeddings (np.ndarray).
    This is the ONLY encoding function — called identically for queries and docs.
    In Stage 3, the audio encoder will replace this for queries only.
    """
    all_embs = []
    for start in tqdm(range(0, len(texts), CONFIG["batch_size"]), desc=desc):
        batch = texts[start : start + CONFIG["batch_size"]]
        enc = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=CONFIG["max_length"],
            return_tensors="pt",
        ).to(DEVICE)
        with torch.no_grad():
            out  = model(**enc)
            embs = mean_pool(out.last_hidden_state, enc["attention_mask"])
            embs = torch.nn.functional.normalize(embs, p=2, dim=-1)
        all_embs.append(embs.cpu().numpy())
    return np.vstack(all_embs)

In [9]:
# ── Cell 8: Encode corpus ─────────────────────────────────────────────────
doc_ids   = list(corpus.keys())
doc_texts = [corpus[d]["contents"] for d in doc_ids]

corpus_embeddings = encode(doc_texts, desc="Encoding corpus")
print(f"Corpus embeddings : {corpus_embeddings.shape}")

Encoding corpus:   0%|          | 0/8 [00:00<?, ?it/s]

Corpus embeddings : (500, 384)


In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
 
import torchaudio
import torchaudio.transforms as T
 
from transformers import (
    HubertModel,
    AutoModel,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)
 
class CFG:
    # models
    audio_model_name  = "facebook/hubert-base-ls960"
    text_model_name   = "sentence-transformers/all-MiniLM-L6-v2"
 
    # embedding dims
    audio_hidden_dim  = 768   # HuBERT-base output
    proj_dim          = 384   # match MiniLM output dim
 
    # audio
    sample_rate       = 16000
    max_audio_len     = 5     # seconds
 
    # hubert unfreezing
    unfreeze_last_n   = 4     # number of HuBERT transformer layers to unfreeze
 
    # training
    batch_size        = 32
    num_epochs        = 10
    lr                = 3e-5
    warmup_steps      = 500
    temperature       = 0.07  # InfoNCE temperature
 
    # loss weights
    lambda_mse        = 0.5
    lambda_cosine     = 1.0
    lambda_infonce    = 2.0
 
    # misc
    seed              = 42
    num_workers       = 2
    device            = "cuda" if torch.cuda.is_available() else "cpu"
    save_path         = "/kaggle/working/audio_encoder_distilled.pt"
 
    # dataset path — update to your Kaggle input path
    dataset_path      = "/kaggle/input/datasets/premsagars2023/medpub-tts/kd_dataset"
 
print(f"Device : {CFG.device}")
print(f"Unfreeze last {CFG.unfreeze_last_n} HuBERT layers")

Device : cuda
Unfreeze last 4 HuBERT layers


In [11]:
class AttentionPooling(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)

    def forward(self, hidden_states):  # (B, T', 768)
        weights = torch.softmax(self.attn(hidden_states), dim=1)  # (B, T', 1)
        return (hidden_states * weights).sum(dim=1)               # (B, 768)

In [12]:
import torch.nn as nn

class AudioEncoder(nn.Module):
    """
    HuBERT backbone with a linear projection head.
    Last N transformer layers are unfrozen; rest frozen.
    Output is L2-normalised, matching teacher embedding space.
    """
    def __init__(self):
        super().__init__()
        self.hubert = HubertModel.from_pretrained(CFG.audio_model_name)
        self.proj = nn.Sequential(
            nn.Linear(CFG.audio_hidden_dim, CFG.audio_hidden_dim),
            nn.GELU(),
            nn.LayerNorm(CFG.audio_hidden_dim),
            nn.Linear(CFG.audio_hidden_dim, CFG.proj_dim),
        )
        self.pool = AttentionPooling(CFG.audio_hidden_dim)
        self._freeze_layers()
 
    def _freeze_layers(self):
        # Freeze everything first
        for param in self.hubert.parameters():
            param.requires_grad = False
 
        # Unfreeze last N transformer layers
        total_layers  = len(self.hubert.encoder.layers)
        unfreeze_from = total_layers - CFG.unfreeze_last_n
        for i, layer in enumerate(self.hubert.encoder.layers):
            if i >= unfreeze_from:
                for param in layer.parameters():
                    param.requires_grad = True
 
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        total     = sum(p.numel() for p in self.parameters())
        print(f"Trainable params: {trainable:,} / {total:,} "
              f"({100*trainable/total:.1f}%) — last {CFG.unfreeze_last_n} layers + proj")
 
    def forward(self, waveforms: torch.Tensor) -> torch.Tensor:
        """
        Args:
            waveforms : (B, T) raw 16kHz audio
        Returns:
            embeddings: (B, proj_dim) L2-normalised
        """
        out    = self.hubert(input_values=waveforms).last_hidden_state  # (B, T', 768)
        pooled = self.pool(out)                                         # (B, 768)
        proj   = self.proj(pooled)                                       # (B, proj_dim)
        return F.normalize(proj, p=2, dim=-1)                           # (B, proj_dim)

In [13]:
def load_audio_encoder(checkpoint_path: str, device: str = CFG.device) -> AudioEncoder:
    """
    Load a trained AudioEncoder from a checkpoint.
 
    Args:
        checkpoint_path : path to .pt file saved during training
        device          : "cuda" or "cpu"
 
    Returns:
        model in eval mode, ready for inference
    """
    ckpt  = torch.load(checkpoint_path, map_location=device)
    model = AudioEncoder().to(device)
    model.load_state_dict(ckpt["model_state"])
    model.eval()
    print(f"Loaded AudioEncoder from '{checkpoint_path}' (epoch {ckpt['epoch']})")
    return model

In [14]:
def encode_audio(
    audio_paths: list[str],
    model: AudioEncoder,
    device: str = CFG.device,
    batch_size: int = 32,
) -> np.ndarray:
    """
    Encode a list of .wav files into L2-normalised embeddings.
    Drop-in replacement for encode() in the baseline notebook.
 
    Args:
        audio_paths : list of paths to 16kHz mono .wav files
        model       : loaded AudioEncoder (from load_audio_encoder)
        device      : "cuda" or "cpu"
        batch_size  : adjust based on VRAM
 
    Returns:
        embeddings  : np.ndarray of shape (N, proj_dim)
 
    Usage in evaluator notebook (Stage 3):
        audio_model      = load_audio_encoder("/kaggle/input/.../audio_encoder_distilled.pt")
        query_embeddings = encode_audio([query_audio[q] for q in qids], audio_model)
    """
    max_samples = CFG.sample_rate * CFG.max_audio_len
    all_embs    = []
 
    model.eval()
    for start in tqdm(range(0, len(audio_paths), batch_size), desc="Encoding audio"):
        batch_paths = audio_paths[start : start + batch_size]
        waveforms   = []
 
        for path in batch_paths:
            waveform, sr = torchaudio.load(path)
            if sr != CFG.sample_rate:
                waveform = T.Resample(sr, CFG.sample_rate)(waveform)
            waveform = waveform.mean(dim=0)  # mono
            if waveform.shape[0] < max_samples:
                waveform = F.pad(waveform, (0, max_samples - waveform.shape[0]))
            else:
                waveform = waveform[:max_samples]
            waveforms.append(waveform)
 
        batch_tensor = torch.stack(waveforms).to(device)  # (B, T)
        with torch.no_grad():
            embs = model(batch_tensor)                     # (B, proj_dim)
        all_embs.append(embs.cpu().numpy())
 
    return np.vstack(all_embs)

In [15]:
audio_model      = load_audio_encoder("/kaggle/input/models/premsagars2023/pubmed-30k-trained/pytorch/default/1/audio_encoder_distilled.pt")
qids             = list(queries.keys())
query_embeddings = encode_audio([query_audio[q] for q in qids], audio_model)

print(f"Query embeddings : {query_embeddings.shape}")

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Trainable params: 29,239,681 / 95,259,905 (30.7%) — last 4 layers + proj
Loaded AudioEncoder from '/kaggle/input/models/premsagars2023/pubmed-30k-trained/pytorch/default/1/audio_encoder_distilled.pt' (epoch 10)


Encoding audio:   0%|          | 0/7 [00:00<?, ?it/s]

Query embeddings : (200, 384)


In [16]:
# ── Cell 10: Retrieval (cosine sim via dot product on L2-normed vecs) ─────
print("Running retrieval ...")

# (num_queries, num_docs)
scores = query_embeddings @ corpus_embeddings.T

# For each query: sorted doc indices descending
top_k       = CONFIG["top_k"]
ranked_idxs = np.argsort(-scores, axis=1)[:, :top_k]

# results: { qid -> [ (doc_id, score), ... ] } top-k
results = {}
for i, qid in enumerate(qids):
    results[qid] = [
        (doc_ids[idx], float(scores[i, idx]))
        for idx in ranked_idxs[i]
    ]

print(f"Retrieval done for {len(results):,} queries, top-{top_k} docs each.")

Running retrieval ...
Retrieval done for 200 queries, top-10 docs each.


In [17]:
# ── Cell 11: Evaluation ───────────────────────────────────────────────────
def compute_metrics(results, qrels, top_k=10):
    """
    Compute MRR, MAP, NDCG@k, Recall@1/5/10.

    Args:
        results : { qid -> [(doc_id, score), ...] }  ranked list
        qrels   : { qid -> {doc_id: relevance} }
        top_k   : cutoff for NDCG and Recall

    Returns:
        dict of metric_name -> float
    """
    mrr_scores, map_scores, ndcg_scores = [], [], []
    recall_at = {1: [], 5: [], 10: []}

    for qid, ranked in results.items():
        relevant = set(qrels.get(qid, {}).keys())
        if not relevant:
            continue

        ranked_ids = [doc_id for doc_id, _ in ranked]

        # ── MRR ──────────────────────────────────────────────────────────
        rr = 0.0
        for rank, doc_id in enumerate(ranked_ids, start=1):
            if doc_id in relevant:
                rr = 1.0 / rank
                break
        mrr_scores.append(rr)

        # ── MAP ──────────────────────────────────────────────────────────
        num_relevant_seen, precision_sum = 0, 0.0
        for rank, doc_id in enumerate(ranked_ids, start=1):
            if doc_id in relevant:
                num_relevant_seen += 1
                precision_sum += num_relevant_seen / rank
        ap = precision_sum / len(relevant) if relevant else 0.0
        map_scores.append(ap)

        # ── NDCG@k ───────────────────────────────────────────────────────
        def dcg(ids, k):
            return sum(
                (qrels[qid].get(doc_id, 0)) / np.log2(rank + 1)
                for rank, doc_id in enumerate(ids[:k], start=1)
            )
        ideal_ids = sorted(relevant, key=lambda d: qrels[qid].get(d, 0), reverse=True)
        idcg = dcg(ideal_ids, top_k)
        ndcg = dcg(ranked_ids, top_k) / idcg if idcg > 0 else 0.0
        ndcg_scores.append(ndcg)

        # ── Recall@k ─────────────────────────────────────────────────────
        for k in recall_at:
            hits = len(set(ranked_ids[:k]) & relevant)
            recall_at[k].append(hits / len(relevant))

    metrics = {
        "MRR"       : float(np.mean(mrr_scores)),
        "MAP"       : float(np.mean(map_scores)),
        f"NDCG@{top_k}": float(np.mean(ndcg_scores)),
        "Recall@1"  : float(np.mean(recall_at[1])),
        "Recall@5"  : float(np.mean(recall_at[5])),
        "Recall@10" : float(np.mean(recall_at[10])),
    }
    return metrics


metrics = compute_metrics(results, qrels, top_k=CONFIG["top_k"])

print("\n" + "="*45)
print(" STAGE 1 — UPPER BOUND (Text → Text)")
print("="*45)
for name, val in metrics.items():
    print(f"  {name:<15} {val:.4f}")
print("="*45)


 STAGE 1 — UPPER BOUND (Text → Text)
  MRR             0.8385
  MAP             0.8385
  NDCG@10         0.8732
  Recall@1        0.7600
  Recall@5        0.9550
  Recall@10       0.9800


In [18]:
# ── Cell 12: Save results ─────────────────────────────────────────────────
output = {
    "stage"  : "3_proposed",
    "encoder": CONFIG["encoder_name"],
    "corpus_size": len(corpus),
    "num_queries": len(queries),
    "top_k"  : CONFIG["top_k"],
    "metrics": metrics,
}

out_path = Path(CONFIG["output_dir"]) / f"stage3_proposed_{CONFIG["corpus_size"]}.json"
with open(out_path, "w") as f:
    json.dump(output, f, indent=2)

print(f"Results saved to: {out_path}")

Results saved to: /kaggle/working/results/stage3_proposed_500.json
